# **BRITISH DYNAMISM: UPDATED FACTS**

## *Data loading and preparation*

In [2]:
# Import packages and set filepaths
import pandas as pd
import numpy as np
import altair as alt
from pathlib import Path
from pandas.api.types import CategoricalDtype
import os
import eco_style 
alt.themes.enable("report")

script_dir = Path.cwd()
import_path = script_dir.parent / "Data"
chart_path = script_dir.parent / "Charts"

export_path = script_dir.parent / "Tables"

In [3]:
# IMPORT DATASETS
population_df = pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='population')
firm_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='firm_dynamics')
job_flows_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='job_flows')
site_dynamics_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='site_dynamics')
cohort_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='cohort_analysis')
growth_rates_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_rates')
growth_cats_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='growth_cats')
prod_df =  pd.read_excel(import_path / 'BSD/29_02_2026_BSD_Dynamism_Stats.xlsx', sheet_name='prod')


In [64]:
# Force data types to numeric where possible, * for diclosive values automatically read the column in as a string
def convert_numeric_columns(df):
    """
    Convert all numeric columns to numeric type, replacing '*' with NaN.
    Preserves non-numeric columns (like year, category names) as they are.
    """
    df_converted = df.copy()
    
    for col in df_converted.columns:
        # Try to convert to numeric, replacing '*' and other errors with NaN
        df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')
    
    return df_converted

# Apply to all dataframes
population_df = convert_numeric_columns(population_df)
firm_dynamics_df = convert_numeric_columns(firm_dynamics_df)
job_flows_df = convert_numeric_columns(job_flows_df)
site_dynamics_df = convert_numeric_columns(site_dynamics_df)
cohort_df = convert_numeric_columns(cohort_df)
growth_rates_df = convert_numeric_columns(growth_rates_df)
growth_cats_df = convert_numeric_columns(growth_cats_df)
prod_df = convert_numeric_columns(prod_df)

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_9828/2220193345.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_converted[col] = pd.to_numeric(df_converted[col].replace('*', np.nan), errors='ignore')


## **FACT 1. Lots of small firms, large employers.**
- Small firms dominate the business population.
- Large firms employ a disproportionate share of the workforce.
- This skew affects interpretation of dynamism.

In [73]:
size_df_2023

,year,dimension,category,n_firms,employment,turnover,avg_turnover_per_employee,avg_age,multi_site_firms,multi_site_emp,Total_employment,Total_firms,Total_turnover,Share_of_employment,Share_of_firms,Share_of_turnover
1075,2023,Size,Large (250+),8167,10580082,2159164801,233,27,5760,9135905,24040893,2502261,4392952022,0.440087,0.003264,0.491507
1076,2023,Size,Medium (50-249),39616,3832403,793176430,198,22,14389,1562880,24040893,2502261,4392952022,0.159412,0.015832,0.180557
1077,2023,Size,Micro (0-9),2228047,5177712,759515482,153,11,8848,47329,24040893,2502261,4392952022,0.215371,0.890414,0.172894
1078,2023,Size,Small (10-49),226431,4450696,681095309,142,17,19316,500636,24040893,2502261,4392952022,0.185130,0.090491,0.155043


In [72]:
# Economic contribution by firm size 2023

size_df = population_df[population_df['dimension']=='Size']
size_df_2023 = size_df[size_df['year']==2023]


size_df_2023['Total_employment'] = size_df_2023['employment'].sum()
size_df_2023['Total_firms'] = size_df_2023['n_firms'].sum()
size_df_2023['Total_turnover'] = size_df_2023['turnover'].sum()

size_df_2023['Share_of_employment'] = size_df_2023['employment']/size_df_2023['Total_employment']
size_df_2023['Share_of_firms'] = size_df_2023['n_firms']/size_df_2023['Total_firms']
size_df_2023['Share_of_turnover'] = size_df_2023['turnover']/size_df_2023['Total_turnover']

firmsize_share_of_activity = size_df_2023.melt(id_vars='category',
                                                value_vars=['Share_of_employment','Share_of_firms','Share_of_turnover'],
                                                value_name='Share of activity')

label_map = {
    'Share_of_employment': 'Employment',
    'Share_of_firms': 'Firms',
    'Share_of_turnover': 'Turnover'
}

firmsize_share_of_activity['variable'] = firmsize_share_of_activity['variable'].map(label_map)

sizeband_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']
variable_order = ['Firms', 'Employment', 'Turnover']

chart = alt.Chart(firmsize_share_of_activity).mark_bar().encode(
    x=alt.X('category:O', sort=sizeband_order),
    y=alt.Y('Share of activity:Q', axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('variable:N', sort=variable_order).legend(title=None, orient='bottom', 
        direction='horizontal'),
    xOffset=alt.XOffset('variable:N', sort=variable_order)
)

display(chart)
#chart.save(chart_path / 'Descriptive paper/Composition/BSD_firmsize_contribution.png', scale_factor=2.0)

/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_9828/162112370.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  size_df_2023['Total_employment'] = size_df_2023['employment'].sum()
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_9828/162112370.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  size_df_2023['Total_firms'] = size_df_2023['n_firms'].sum()
/var/folders/dp/dmj0l1fj4cld5dkvtqpy9bqc0000gn/T/ipykernel_9828/162112370.py:9: SettingWithCopyWarning: 
A value is trying to be set

alt.Chart(...)

## **FACT 2. Young firms are playing a smaller role.**
- Young firms are central to a dynamic economy.
- The share of young firms in the UK has declined since 2017.
- The share of employment in young firms has also fallen in recent years, though by less.
- Young firms are hiring fewer people.

In [ ]:
# Contribution of young firms (firm share and employment share)
age_population_df = population_df[population_df['dimension']=='Age']

age_population_df = age_population_df[age_population_df['year'] >=2004]

# Combine new and young firms to create new 0-5 category
age_population_df['age_category'] = age_population_df['category'].replace({'New (0-2 years)': 'Young (0-5 years)', 'Young (3-5 years)': 'Young (0-5 years)'})
age_population_df = age_population_df.groupby(['year', 'dimension', 'age_category']).agg({'n_firms':'sum', 'employment':'sum', 'turnover':'sum'}).reset_index()

# Calculate share of total firms, employment, and turnover for each age category
age_population_df['total_firms'] = age_population_df.groupby('year')['n_firms'].transform('sum')        
age_population_df['total_employment'] = age_population_df.groupby('year')['employment'].transform('sum')
age_population_df['total_turnover'] = age_population_df.groupby('year')['turnover'].transform('sum')

age_population_df['share_of_firms'] = age_population_df['n_firms'] / age_population_df['total_firms']
age_population_df['share_of_employment'] = age_population_df['employment'] / age_population_df['total_employment']
age_population_df['share_of_turnover'] = age_population_df['turnover'] / age_population_df['total_turnover']

# Plot just the young firm share
young_firm_contribution_df = age_population_df[age_population_df['age_category']=='Young (0-5 years)']
young_firm_chart = alt.Chart(young_firm_contribution_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0)),
    y=alt.Y('share_of_firms:Q', axis=alt.Axis(format='%'), title='Share of firms that are young (0-5 years)'),
)       

young_emp_chart = alt.Chart(young_firm_contribution_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 != 0 ? datum.label : ''",
             labelAngle=0)),
    y=alt.Y('share_of_employment:Q', axis=alt.Axis(format='%'), title='Share of employment in young firms (0-5 years)'),
) 

young_activity_chart = young_firm_chart | young_emp_chart
display(young_activity_chart)

#young_activity_chart.save(chart_path / 'Descriptive paper/Composition/BSD_young_firm_contribution.png', scale_factor=2.0)

## **FACT 3. Stable entry and exit.**
- Entry and exit are the clearest signals of dynamism.
- Entry and exit rates vary year by year but show no clear time trend at a UK-wide level.
- The COVID pandemic saw net entry come to an end.
- The rate of entry and exit differs across sectors.
- Differences across sectors have narrowed over time.

In [65]:
# Calculate entry and exit rates from absolute numbers
firm_dynamics_df = firm_dynamics_df.sort_values(['category','dimension','year'])

firm_dynamics_df['total_firms_lag'] = firm_dynamics_df.groupby(['category','dimension'])['n_firms'].shift(1)

firm_dynamics_df['entry_rate'] = (firm_dynamics_df['n_entrants'] + firm_dynamics_df['n_entry_and_exit']) / firm_dynamics_df['total_firms_lag']
firm_dynamics_df['exit_rate'] = (firm_dynamics_df['n_exiters'] + firm_dynamics_df['n_entry_and_exit']) / firm_dynamics_df['total_firms_lag']


In [6]:
# Chart 1: ENTRY AND EXIT RATES
total_firm_dynamics_df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Total']
total_firm_dynamics_df = total_firm_dynamics_df[total_firm_dynamics_df['year'] >=1999]

total_entry_exit_df = total_firm_dynamics_df.melt(id_vars='year',value_vars=['entry_rate','exit_rate'])

total_entry_exit_df['variable'] = total_entry_exit_df['variable'].map({
    'entry_rate': 'Entry rate',
    'exit_rate': 'Exit rate'
})

entry_exit_chart = alt.Chart(total_entry_exit_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''",  # Show every 2nd year
            labelAngle=0), title=None),
    y=alt.Y('value:Q', axis=alt.Axis(format='%')),
    color=alt.Color('variable:N', legend=None)
).properties(
    width=600,
    height=400
)
# End labels
end_labels = alt.Chart(
    total_entry_exit_df[total_entry_exit_df['year'] == total_entry_exit_df['year'].max()]
).mark_text(
    align='left', dx=8, fontSize=12
).encode(
    x='year:O',
    y='value:Q',
    text=alt.Text('variable:N'),
    color=alt.Color('variable:N', legend=None)
)

chart = (entry_exit_chart + end_labels)
chart
# chart.save(chart_path / 'Descriptive paper/Dynamism/BSD_entry_exit_rates.png', scale_factor=2.0)

alt.LayerChart(...)

In [ ]:

# ── Focus classification: "Declining", "Rising", or "Other" ──────────────────
focus_map = {
    "Business Support Services": "Declining",
    "IT & Computer Services":    "Declining",
    "Transport & Logistics":     "Rising",
    "Retail Trade":              "Rising",
}

# ── Prepare data ──────────────────────────────────────────────────────────────
periods = {
    "2000-2007": (2000, 2007),
    "2008-2015": (2008, 2015),
    "2016-2022": (2016, 2022),
}

df = firm_dynamics_df[firm_dynamics_df["dimension"] == "sector"]

sector_churn = (
    df.groupby("category")[["entry_rate", "exit_rate"]]
    .mean()
    .assign(net_churn=lambda x: x["entry_rate"] + x["exit_rate"])
    .sort_values("net_churn", ascending=False)  # ascending so top = highest on chart
)
sector_order = sector_churn.index.tolist()

rows = []
for sector in sector_order:
    df_sector = df[df["category"] == sector]
    row = {"Industry": sector}
    for period_label, (year_start, year_end) in periods.items():
        df_period = df_region[
            (df_region["year"] >= year_start) & (df_region["year"] <= year_end)
        ]
        row[period_label] = round(df_period["entry_rate"].mean() * 100, 1)
    rows.append(row)

df_wide = pd.DataFrame(rows)
df_wide["Focus"] = df_wide["Industry"].map(focus_map).fillna("Other")

period_order = list(periods.keys())

df_long = df_wide.melt(
    id_vars=["Industry", "Focus"],
    value_vars=period_order,
    var_name="Period",
    value_name="Entry Rate (%)",
)

# ── Chart ─────────────────────────────────────────────────────────────────────
chart = (
    alt.Chart(df_long)
    .mark_bar()
    .encode(
        y=alt.Y(
            "Industry:N",
            sort=region_order,
            axis=alt.Axis(title=None, labelFontSize=11, labelColor="#475569"),
        ),
        x=alt.X(
            "Entry Rate (%):Q",
            axis=alt.Axis(
                title="Entry Rate (%)",
                grid=True,
                gridColor="#E2E8F0",
                tickCount=5,
                labelColor="#94A3B8",
                titleColor="#64748B",
                titleFontSize=12,
            ),
            scale=alt.Scale(domain=[0, 35]),
        ),
        yOffset=alt.YOffset("Period:N", sort=period_order),
        color=alt.Color(
            "Period:N",
            sort=period_order,
            scale=alt.Scale(
                domain=period_order,
                range=["#93C5FD", "#3B82F6", "#1E3A8A"],  # blue light → dark
            ),
            legend=alt.Legend(
                title=None,
                orient="top",
                labelFontSize=11,
                labelColor="#475569",
                symbolSize=80,
            ),
        ),
        tooltip=[
            alt.Tooltip("Industry:N",       title="Industry"),
            alt.Tooltip("Period:N",         title="Period"),
            alt.Tooltip("Entry Rate (%):Q", title="Entry Rate (%)", format=".1f"),
        ],
    )
    .properties(
        width=600,
        height=400,
    )
    .configure_view(stroke=None)
    .configure_axis(labelFont="Arial", titleFont="Arial")
    .configure_title(font="Arial")
)

display(chart)
chart.save("entry_rates_chart.html")
chart.save(chart_path / 'Descriptive paper/Dynamism/region_entry.png', scale_factor=2)
print("Saved: entry_rates_chart.html")

alt.Chart(...)

Saved: entry_rates_chart.html


** SUPPLEMENTARY BREAKDOWN PLOTS FOR THE DATA ANNEX**

In [ ]:
# ENTRY AND EXIT FACETED BY INDUSTRIES FOR ANNEX
sector_firm_dynamics_df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Sector']
sector_firm_dynamics_df = sector_firm_dynamics_df[sector_firm_dynamics_df['year'] >= 1999]

sector_entry_exit_df = sector_firm_dynamics_df.melt(
    id_vars=['year', 'category'],  # include category so it survives the melt
    value_vars=['entry_rate', 'exit_rate']
)

sector_entry_exit_df['variable'] = sector_entry_exit_df['variable'].map({
    'entry_rate': 'Entry rate',
    'exit_rate': 'Exit rate'
})

max_year = int(sector_entry_exit_df['year'].max())

color_scale = alt.Scale(
    domain=['Entry rate', 'Exit rate'],
    range=[eco_style.pallete['nominal_1'], eco_style.pallete['nominal_2']]
)

line = alt.Chart().mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
        labelExpr="datum.value % 4 == 0 ? datum.label : ''",
        labelAngle=0), title=None),
    y=alt.Y('value:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False)),
    color=alt.Color('variable:N', scale=color_scale, legend=None)
)

end_labels = alt.Chart().transform_filter(
    alt.datum.year == max_year
).mark_text(align='left', dx=5, fontSize=9).encode(
    x='year:O',
    y='value:Q',
    text='variable:N',
    color=alt.Color('variable:N', scale=color_scale, legend=None)
)

# Key: use alt.layer(...).facet(data=...) — NOT chart + labels then .facet()
sector_entry_exit_chart = alt.layer(line, end_labels).facet(
    data=sector_entry_exit_df,
    facet=alt.Facet('category:N', title=None),
    columns=3
).resolve_scale(y='independent')

sector_entry_exit_chart.save(chart_path / 'Descriptive paper/Annex/BSD_sector_entry_exit_rates.png', scale_factor=2.0)


In [ ]:
# ENTRY AND EXIT FACETED BY REGIONS FOR ANNEX
region_firm_dynamics_df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Region']
region_firm_dynamics_df = region_firm_dynamics_df[region_firm_dynamics_df['year'] >= 1999]

region_entry_exit_df = region_firm_dynamics_df.melt(
    id_vars=['year', 'category'],  # include category so it survives the melt
    value_vars=['entry_rate', 'exit_rate']
)

region_entry_exit_df['variable'] = region_entry_exit_df['variable'].map({
    'entry_rate': 'Entry rate',
    'exit_rate': 'Exit rate'
})

max_year = int(region_entry_exit_df['year'].max())

color_scale = alt.Scale(
    domain=['Entry rate', 'Exit rate'],
    range=[eco_style.pallete['nominal_1'], eco_style.pallete['nominal_2']]
)

line = alt.Chart().mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
        labelExpr="datum.value % 4 == 0 ? datum.label : ''",
        labelAngle=0), title=None),
    y=alt.Y('value:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False)),
    color=alt.Color('variable:N', scale=color_scale, legend=None)
)

end_labels = alt.Chart().transform_filter(
    alt.datum.year == max_year
).mark_text(align='left', dx=5, fontSize=9).encode(
    x='year:O',
    y='value:Q',
    text='variable:N',
    color=alt.Color('variable:N', scale=color_scale, legend=None)
)

# Key: use alt.layer(...).facet(data=...) — NOT chart + labels then .facet()
region_entry_exit_chart = alt.layer(line, end_labels).facet(
    data=region_entry_exit_df,
    facet=alt.Facet('category:N', title=None),
    columns=3
).resolve_scale(y='independent')

region_entry_exit_chart.save(chart_path / 'Descriptive paper/Annex/BSD_region_entry_exit_rates.png', scale_factor=2.0)


alt.FacetChart(...)

## **FACT 4. Declining exit for large firms.**
- The smallest firms are sustaining the exit rate at the whole-economy level.

In [ ]:
# EXIT RATE BY SIZE OF FIRM (EMPLOYMENT)

size_firm_dynamics_df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Size']
size_firm_dynamics_df = size_firm_dynamics_df[size_firm_dynamics_df['year'] >= 1999]

size_exit = size_firm_dynamics_df[['year','category','exit_rate']]

# Get last year for end labels
last_year = size_exit['year'].max()
end_labels = size_exit[size_exit['year'] == last_year]

end_labels.loc[end_labels['category'] == 'Large (250+)', 'exit_rate'] -= 0.0025
end_labels.loc[end_labels['category'] == 'Medium (50-249)', 'exit_rate'] += 0.0025

lines = alt.Chart(size_exit).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''",
            labelAngle=0)),
    y=alt.Y('exit_rate:Q', axis=alt.Axis(format='%')),
    color=alt.Color('category:N',
                     scale=alt.Scale(
                         domain=['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)'],
                         range=['#eb5c2e', 'rgba(24, 42, 56, 0.4)', 'rgba(24, 42, 56, 0.7)', '#122b39']
                     ),
                     legend=None)
).properties(width=600, height=400)

labels = alt.Chart(end_labels).mark_text(align='left', dx=5, fontSize=11).encode(
    x=alt.X('year:O'),
    y=alt.Y('exit_rate:Q'),
    text='category:N',
    color=alt.Color('category:N',
                     scale=alt.Scale(
                         domain=['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)'],
                         range=['#eb5c2e', 'rgba(24, 42, 56, 0.9)', 'rgba(24, 42, 56, 0.7)', '#122b39']
                     ))
)

size_exit_chart = (lines + labels)

display(size_exit_chart)

size_exit_chart.save(chart_path / 'Descriptive paper/Dynamism/BSD_exit_rates_by_size.png', scale_factor=2.0)



## **FACT 5. The least productive firms are exiting.**

- In a dynamic economy, the worst performing firms should be leaving the market.
- Firms with the lowest productivity, against industry peers, consistently have the highest rate of exit.


In [ ]:
# FIRM EXIT BY PRODUCTIVITY WITH MIDDLE GROUPS COMBINED
prod_firm_dynamics_df = firm_dynamics_df[firm_dynamics_df['dimension'] == 'Productivity']
prod_firm_dynamics_df = prod_firm_dynamics_df[prod_firm_dynamics_df['year'] >= 1999].copy()

# Combine middle categories
prod_firm_dynamics_df['category'] = prod_firm_dynamics_df['category'].replace({
    'Low-Median (P10-P50)': 'Middle (P10-P90)',
    'High-Median (P50-P90)': 'Middle (P10-P90)'
})

# Average the exit rates for the combined middle group
prod_firm_dynamics_df = (
    prod_firm_dynamics_df
    .groupby(['year', 'category'])['exit_rate']
    .mean()
    .reset_index()
)

color_scale = alt.Scale(
    domain=['Laggards (<P10)', 'Middle (P10-P90)', 'Frontier (P90+)'],
    range=['#e6224b', '#122b39', "#179fdb"]
)

line = alt.Chart(prod_firm_dynamics_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''",
            labelAngle=0)),
    y=alt.Y('exit_rate:Q', axis=alt.Axis(format='%')),
    color=alt.Color('category:N', scale=color_scale, legend=None)
).properties(height=600, width=800)

last_points = prod_firm_dynamics_df.loc[
    prod_firm_dynamics_df.groupby('category')['year'].idxmax()
].copy()

labels = alt.Chart(last_points).mark_text(
    align='left',
    dx=5,
    fontSize=11,
    fontWeight='bold'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('exit_rate:Q'),
    text='category:N',
    color=alt.Color('category:N', scale=color_scale, legend=None)
)

prod_exit_chart = (line + labels).properties(
    height=400,
    width=600
)

display(prod_exit_chart)
prod_exit_chart.save(chart_path / 'Descriptive paper/Dynamism/BSD_exit_rates_by_productivity.png')

## **FACT 6. New firms create fewer jobs.**

- New firms are being born smaller.
- Firms may be needing fewer hands to get off the ground. 

In [ ]:
# Average employment at entry over time

# Use cohort table for this and plot avg_size at age 0 across cohorts

size_at_entry = cohort_df[cohort_df['age']==0]

size_at_entry_chart = alt.Chart(size_at_entry).mark_line().encode(
    x=alt.X('cohort:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''", 
            labelAngle=0)),
    y=alt.Y('avg_size:Q')
).properties(height=400, width=600)

display(size_at_entry_chart)
size_at_entry_chart.save(chart_path / 'average_size_at_entry.png')

## **FACT 7. Longer-lived firms.**

- The survival of firms tells us about the business environment.
- The risk of failure for firms is highest in the earliest years.
- Firms starting up after the financial crisis are more likely to survive longer than those entering before it.
- Entrants during the COVID pandemic faced a particularly challenging environment.
- The effect of increased survival on dynamism is ambiguous.

In [ ]:
# KM survival by cohort entry period
cohort_s = cohort[(cohort['age'] > 0) & (cohort['age'] <= 10)].copy()
cohort_s['period'] = cohort_s['cohort'].apply(
    lambda x: 'Pre-GFC (1999–2007)'  if x <= 2007 else
              'Post-GFC entry (2008–2019)' if x <= 2019 else
              'COVID era (2020+)')

avg_period = cohort_s.groupby(['period', 'age'])['kaplan_meier_rate'].mean().reset_index()

alt.Chart(avg_period).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('age:Q', title='Firm age (years)'),
    y=alt.Y('kaplan_meier_rate:Q', title='Avg KM survival estimate', scale=alt.Scale(zero=False)),
    color=alt.Color('period:N', legend=alt.Legend(title='Cohort entry period')),
    tooltip=['period:N', 'age:Q', alt.Tooltip('kaplan_meier_rate:Q', format='.2f')]
).properties(title='KM survival estimates by cohort entry period', width=620, height=300)


## **FACT 8. Lower job reallocation.**

- Job reallocation is the constant churn of positions created and destroyed as firms enter, exit, grow, and shrink.
- The total job reallocation rate in the UK has declined over the past two decades.
- The decline is largest when looking at jobs associated with entry and exit.
- Exiting firms are taking away fewer jobs.
- Incumbent firms are creating less jobs.
- Incumbent firms appear responsive to macroeconomics trends.

In [ ]:
# AGGREGATE: Job destruction from exiters vs contracting incumbents

total_job_flows_df = job_flows_df[job_flows_df['dimension'] == 'Total']
total_job_flows_df = total_job_flows_df[total_job_flows_df['year'] >= 1999]

entry_exit_job_flows_df = total_job_flows_df.melt(
    id_vars=['year'], 
    value_vars=['jc_rate_entrants', 'jd_rate_exiters'], 
    var_name='job_flow_type', 
    value_name='job_flow_rate'
)

# Map variable names
label_map = {
    'jc_rate_entrants': 'Job creation',
    'jd_rate_exiters': 'Job destruction'
}
entry_exit_job_flows_df['job_flow_type'] = entry_exit_job_flows_df['job_flow_type'].map(label_map)

color_scale = alt.Scale(
    domain=['Job creation', 'Job destruction'],
    range=['#179fdb', '#e6224b']
)

last_points = entry_exit_job_flows_df.loc[
    entry_exit_job_flows_df.groupby('job_flow_type')['year'].idxmax()
].copy()

lines = alt.Chart(entry_exit_job_flows_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''",
            labelAngle=0)),
    y=alt.Y('job_flow_rate:Q', axis=alt.Axis(format='%'), title='Entry and exit'),
    color=alt.Color('job_flow_type:N', scale=color_scale, legend=None)
).properties(height=500, width=600)

labels = alt.Chart(last_points).mark_text(
    align='left',
    dx=5,
    fontSize=11,
    fontWeight='bold'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('job_flow_rate:Q'),
    text='job_flow_type:N',
    color=alt.Color('job_flow_type:N', scale=color_scale, legend=None)
)

entry_exit_job_flow_chart = (lines + labels)
entry_exit_job_flow_chart

In [ ]:
# AGGREGATE: Job creation from entrants vs expanding incumbents
total_job_flows_df = job_flows_df[job_flows_df['dimension'] == 'Total']
total_job_flows_df = total_job_flows_df[total_job_flows_df['year'] >= 1999]

incumbent_job_flows_df = total_job_flows_df.melt(
    id_vars=['year'], 
    value_vars=['jc_rate_incumbents', 'jd_rate_incumbents'], 
    var_name='job_flow_type', 
    value_name='job_flow_rate'
)

# Map variable names
label_map = {
    'jc_rate_incumbents': 'Job creation',
    'jd_rate_incumbents': 'Job destruction'
}
incumbent_job_flows_df['job_flow_type'] = incumbent_job_flows_df['job_flow_type'].map(label_map)

color_scale = alt.Scale(
    domain=['Job creation', 'Job destruction'],
    range=['#179fdb', '#e6224b']
)

last_points = incumbent_job_flows_df.loc[
    incumbent_job_flows_df.groupby('job_flow_type')['year'].idxmax()
].copy()

lines = alt.Chart(incumbent_job_flows_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(
                labelExpr="datum.value % 2 == 0 ? datum.label : ''",
            labelAngle=0)),
    y=alt.Y('job_flow_rate:Q', axis=alt.Axis(format='%'), title='Incumbents'),
    color=alt.Color('job_flow_type:N', scale=color_scale, legend=None)
).properties(height=500, width=600)

labels = alt.Chart(last_points).mark_text(
    align='left',
    dx=5,
    fontSize=11,
    fontWeight='bold'
).encode(
    x=alt.X('year:O'),
    y=alt.Y('job_flow_rate:Q'),
    text='job_flow_type:N',
    color=alt.Color('job_flow_type:N', scale=color_scale, legend=None)
)

incumbent_job_flow_chart = (lines + labels)
incumbent_job_flow_chart

In [ ]:
# JOB FLOW COMPARISON BY MARGIN
job_flow_comparison = entry_exit_job_flow_chart | incumbent_job_flow_chart
job_flow_comparison

job_flow_comparison.save(chart_path / 'Descriptive paper/Dynamism/BSD_job_flow_comparison.png', scale_factor=2.0)

## **FACT 9. Firm growth is less extreme.**

In [ ]:
# DHS growth rate distribution (excluding Micro firms)
total_growth_df = growth_rates_df[growth_rates_df['dimension'] == 'Size'].copy()
total_growth_df = total_growth_df[total_growth_df['category'] != 'Micro (0-9)']
total_growth_df = total_growth_df[total_growth_df['year'] >= 2003]
total_growth_df = total_growth_df.groupby('year').agg({
    'mean_dhs_growth': 'mean',
    'p10_dhs_growth': 'mean',
    'p90_dhs_growth': 'mean'
}).reset_index()

# Keep a wide version for the shaded band
band_df = total_growth_df[['year', 'p10_dhs_growth', 'p90_dhs_growth']].copy()

# Melt for the lines
total_growth_df = total_growth_df.melt(
    id_vars='year',
    value_vars=['mean_dhs_growth', 'p10_dhs_growth', 'p90_dhs_growth'],
    var_name='statistic', value_name='growth_rate'
)

label_map = {'mean_dhs_growth': 'Mean', 'p10_dhs_growth': 'P10', 'p90_dhs_growth': 'P90'}
total_growth_df['label'] = total_growth_df['statistic'].map(label_map)

max_year = total_growth_df['year'].max()

# Colour scheme: dark navy for mean, muted tones for P10/P90
color_scale = alt.Scale(
    domain=['mean_dhs_growth', 'p10_dhs_growth', 'p90_dhs_growth'],
    range=['#1b3a4b', '#a3b8c8', '#a3b8c8']
)

# Shaded band between P10 and P90
band = alt.Chart(band_df).mark_area(opacity=0.08, color='#1b3a4b').encode(
    x=alt.X('year:O', axis=alt.Axis(
        labelExpr="datum.value % 2 == 0 ? datum.label : ''",
        labelAngle=0, title=None)),
    y=alt.Y('p10_dhs_growth:Q'),
    y2=alt.Y2('p90_dhs_growth:Q')
)

# Lines
lines = alt.Chart(total_growth_df).mark_line(strokeWidth=2).encode(
    x=alt.X('year:O'),
    y=alt.Y('growth_rate:Q'),
    color=alt.Color('statistic:N', scale=color_scale, legend=None),
    strokeDash=alt.StrokeDash(
        'statistic:N',
        scale=alt.Scale(
            domain=['mean_dhs_growth', 'p10_dhs_growth', 'p90_dhs_growth'],
            range=[[0], [4, 2], [4, 2]]  # solid for mean, dashed for P10/P90
        ),
        legend=None
    )
)

# End labels
end_labels = alt.Chart(total_growth_df).transform_filter(
    alt.datum.year == max_year
).mark_text(align='left', dx=5, fontSize=11).encode(
    x=alt.X('year:O'),
    y=alt.Y('growth_rate:Q'),
    text=alt.Text('label:N'),
    color=alt.Color('statistic:N', scale=color_scale, legend=None)
)

total_dhs_growth_chart = (band + lines + end_labels).properties(
    width=500,
    height=300,
)

display(total_dhs_growth_chart)
total_dhs_growth_chart.save(chart_path / 'Descriptive paper/Dynamism/total_dhs_growth_distribution_excluding_micro.png', scale_factor=2.0)

## **FACT 10. Increased prevalence of stagnant firms.**

In [50]:
# First lets assess the prevlance of stagnant firms
growth_cats_df['stagnant_firm_share'] = growth_cats_df['n_stagnant'] / growth_cats_df['n_incumbents']
growth_cats_df['stagnant_emp_share'] = growth_cats_df['stagnant_emp'] / growth_cats_df['employment']

stagnant_emp_share_2010 = (
    growth_cats_df
    .query("year == 2010")
    ['stagnant_emp_share']
    .mean()
)

stagnant_emp_share_2022= (
    growth_cats_df
    .query("year == 2022")
    ['stagnant_emp_share']
    .mean()
)
print(f"Stagnant emp share 2010: {stagnant_emp_share_2010:.1%}")
print(f"Stagnant emp share 2022: {stagnant_emp_share_2022:.1%}")


Stagnant emp share 2010: 32.1%
Stagnant emp share 2022: 49.1%


In [38]:
# Stagnant firm share by size band
size_stagnant_df = (
    growth_cats_df[growth_cats_df['dimension'] == 'Size']
    .copy()
)
size_stagnant_df['stagnant_firm_share'] = size_stagnant_df['n_stagnant'] / size_stagnant_df['n_incumbents']

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']

avg_by_size = (
    size_stagnant_df
    .groupby('category')['stagnant_firm_share']
    .mean()
    .reindex(size_order)
)

for size, share in avg_by_size.items():
    print(f"{size}: {share:.1%}")

Micro (0-9): 65.2%
Small (10-49): 48.9%
Medium (50-249): 38.5%
Large (250+): 26.5%


In [42]:
# Of all stagnant firms, what share are each size band?
size_stagnant_df = (
    growth_cats_df[growth_cats_df['dimension'] == 'Size']
    .copy()
)

# Sum n_stagnant across size bands per year, then calculate each size's share
size_stagnant_df['total_stagnant'] = size_stagnant_df.groupby('year')['n_stagnant'].transform('sum')
size_stagnant_df['stagnant_size_share'] = size_stagnant_df['n_stagnant'] / size_stagnant_df['total_stagnant']

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']

avg_by_size = (
    size_stagnant_df
    .groupby('category')['stagnant_size_share']
    .mean()
    .reindex(size_order)
)

for size, share in avg_by_size.items():
    print(f"{size}: {share:.1%}")


Micro (0-9): 90.8%
Small (10-49): 8.0%
Medium (50-249): 1.1%
Large (250+): 0.2%


In [53]:
# Of all stagnant employment, what share are each size band?
size_stagnant_df = (
    growth_cats_df[growth_cats_df['dimension'] == 'Size']
    .copy()
)

# Sum n_stagnant across size bands per year, then calculate each size's share
size_stagnant_df['total_stagnant_emp'] = size_stagnant_df.groupby('year')['stagnant_emp'].transform('sum')
size_stagnant_df['stagnant_size_emp_share'] = size_stagnant_df['stagnant_emp'] / size_stagnant_df['total_stagnant_emp']

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']

avg_by_size = (
    size_stagnant_df
    .groupby('category')['stagnant_size_emp_share']
    .mean()
    .reindex(size_order)
)

for size, share in avg_by_size.items():
    print(f"{size}: {share:.1%}")


Micro (0-9): 30.2%
Small (10-49): 21.8%
Medium (50-249): 13.8%
Large (250+): 34.2%


In [46]:
# PLOT SHARE OF STAGNANT FIRMS AGAINT EMPLOYMENT SHARE IN STAGNANT FIRMS

growth_cats_df['stagnant_firm_share'] = growth_cats_df['n_stagnant'] / growth_cats_df['n_incumbents']
growth_cats_df['stagnant_emp_share'] = growth_cats_df['stagnant_emp'] / growth_cats_df['employment']

total_growth_cats_df = growth_cats_df[growth_cats_df['dimension']=='Total']
total_growth_cats_df = total_growth_cats_df[total_growth_cats_df['year']>=1999]

stagnant_chart_df = total_growth_cats_df.melt(id_vars='year', value_vars=['stagnant_firm_share','stagnant_emp_share'])

# Reformat variable labels
label_map = {
    'stagnant_firm_share': 'Firm share',
    'stagnant_emp_share': 'Employment share'
}
stagnant_chart_df['variable'] = stagnant_chart_df['variable'].map(label_map)

color_scale = alt.Scale(
    domain=['Firm share', 'Employment share'],
    range=[eco_style.pallete['nominal_1'], eco_style.pallete['nominal_5']]
)

# Base line chart
line = alt.Chart(stagnant_chart_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 == 0 ? datum.label : ''",
            labelAngle=0)),
    y=alt.Y('value:Q', axis=alt.Axis(format='%'), title='Prevalence of stagnant firms (no employment change)'),
    color=alt.Color('variable:N', title='', scale=color_scale, legend=None)
)

# End labels — filter to final year only
end_labels = alt.Chart(stagnant_chart_df[stagnant_chart_df['year'] == stagnant_chart_df['year'].max()]).mark_text(
    align='left',
    dx=6,
    fontSize=11
).encode(
    x=alt.X('year:O'),
    y=alt.Y('value:Q'),
    text=alt.Text('variable:N'),
    color=alt.Color('variable:N', scale=color_scale, legend=None)
)

stagnation_chart = (line + end_labels).properties(
    height=400,
    width=500
)

stagnation_chart

alt.LayerChart(...)

In [ ]:
# STAGNANT FIRMS — FACETED BY MEASURE, LINES BY FIRM SIZE

size_growth_cats_df = growth_cats_df[growth_cats_df['dimension'] == 'Size'].copy()
size_growth_cats_df = size_growth_cats_df[size_growth_cats_df['year'] >= 1999]

size_growth_cats_df['stagnant_firm_share'] = size_growth_cats_df['n_stagnant'] / size_growth_cats_df['n_incumbents']
size_growth_cats_df['stagnant_emp_share'] = size_growth_cats_df['stagnant_emp'] / size_growth_cats_df['employment']

stagnant_size_df = size_growth_cats_df.melt(
    id_vars=['year', 'category'],
    value_vars=['stagnant_firm_share', 'stagnant_emp_share']
).assign(variable=lambda df: df['variable'].map({
    'stagnant_firm_share': 'Firm share',
    'stagnant_emp_share': 'Employment share'
}))

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']
max_year = int(stagnant_size_df['year'].max())

color_scale = alt.Scale(
    domain=size_order,
    range=['#eb5c2e', 'rgba(24, 42, 56, 0.4)', 'rgba(24, 42, 56, 0.7)', '#122b39']
)

line = alt.Chart().mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('value:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', scale=color_scale, sort=size_order, legend=None)
)

end_labels = alt.Chart().mark_text(
    align='left', dx=5, fontSize=9
).transform_filter(
    alt.datum.year == max_year
).encode(
    x=alt.X('year:O'),
    y=alt.Y('value:Q'),
    text='category:N',
    color=alt.Color('category:N', scale=color_scale, sort=size_order, legend=None)
)

stagnation_size_chart = alt.layer(
    line, end_labels
).facet(
    data=stagnant_size_df,
    facet=alt.Facet('variable:N', sort=['Firm share', 'Employment share'], title=None),
    columns=2
)

stagnation_size_chart

In [31]:
# STAGNANT EMPLOYMENT SHARE BY FIRM SIZE

size_order = ['Micro (0-9)', 'Small (10-49)', 'Medium (50-249)', 'Large (250+)']
max_year = int(stagnant_size_df['year'].max())

emp_size_df = stagnant_size_df[stagnant_size_df['variable'] == 'Employment share'].copy()

color_scale = alt.Scale(
    domain=size_order,
    range=[
        '#eb5c2e',                  # Micro — orange
        'rgba(24, 42, 56, 0.4)',    # Small — light dark blue
        'rgba(24, 42, 56, 0.7)',    # Medium — mid dark blue
        '#122b39',                  # Large — full dark blue
    ]
)

line = alt.Chart().mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('value:Q', axis=alt.Axis(format='%'), scale=alt.Scale(zero=False),
            title='Employment share in stagnant firms'),
    color=alt.Color('category:N', scale=color_scale, sort=size_order, legend=None)
)

end_labels = alt.Chart().mark_text(
    align='left', dx=5, fontSize=10
).transform_filter(
    alt.datum.year == max_year
).encode(
    x=alt.X('year:O'),
    y=alt.Y('value:Q'),
    text='category:N',
    color=alt.Color('category:N', scale=color_scale, sort=size_order, legend=None)
)

emp_stagnation_size_chart = alt.layer(
    line, end_labels,
    data=emp_size_df
).properties(height=400, width=600)

emp_stagnation_size_chart

alt.LayerChart(...)

In [68]:
# STAGNANT EMPLOYMENT SHARE BY FIRM AGE
# 4 categories — single panel, 4 lines

age_growth_cats_df = growth_cats_df[growth_cats_df['dimension'] == 'Age'].copy()
age_growth_cats_df = age_growth_cats_df[age_growth_cats_df['year'] >= 1999]
age_growth_cats_df['stagnant_emp_share'] = age_growth_cats_df['stagnant_emp'] / age_growth_cats_df['employment']

age_order = ['New (0-2 years)', 'Young (3-5 years)', 'Mature (6-10 years)', 'Old (11+ years)']
max_year = int(age_growth_cats_df['year'].max())

# Warm to dark: youngest firms warmest
color_scale = alt.Scale(
    domain=age_order,
    range=['#e6224b', '#eb5c2e', '#36b7b4', '#122b39']
)

line = alt.Chart(age_growth_cats_df).mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 2 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('stagnant_emp_share:Q', axis=alt.Axis(format='%'), scale=alt.Scale(zero=False)),
    color=alt.Color('category:N', scale=color_scale, sort=age_order, legend=None)
)

end_labels = alt.Chart(age_growth_cats_df[age_growth_cats_df['year'] == max_year]).mark_text(
    align='left', dx=5, fontSize=10
).encode(
    x=alt.X('year:O'),
    y=alt.Y('stagnant_emp_share:Q'),
    text='category:N',
    color=alt.Color('category:N', scale=color_scale, sort=age_order, legend=None)
)

emp_stagnation_age_chart = (line + end_labels).properties(height=400, width=600)
emp_stagnation_age_chart.save(chart_path / 'Descriptive paper/Annex/stagnant_emp_age.png', scale_factor=2)

In [70]:
# STAGNANT EMPLOYMENT SHARE BY SECTOR
# 15 categories — faceted by sector, single line per panel

sector_growth_cats_df = growth_cats_df[growth_cats_df['dimension'] == 'Sector'].copy()
sector_growth_cats_df = sector_growth_cats_df[sector_growth_cats_df['year'] >= 1999]
sector_growth_cats_df['stagnant_emp_share'] = sector_growth_cats_df['stagnant_emp'] / sector_growth_cats_df['employment']

max_year = int(sector_growth_cats_df['year'].max())

# Order regions by average employment share (highest first)
sector_order = (
    sector_growth_cats_df.groupby('category')['stagnant_emp_share']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

line = alt.Chart().mark_line(color=eco_style.pallete['nominal_1']).encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 4 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('stagnant_emp_share:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False))
)

emp_stagnation_sector_chart = alt.layer(line).facet(
    data=sector_growth_cats_df,
    facet=alt.Facet('category:N', sort=sector_order, title=None),
    columns=3
).properties(title='Employment share in stagnant firms by sector')

emp_stagnation_sector_chart

alt.FacetChart(...)

In [71]:
# STAGNANT FIRM SHARE AND EMPLOYMENT SHARE BY SECTOR
# 15 categories — faceted by sector, two lines per panel

sector_growth_cats_df2 = growth_cats_df[growth_cats_df['dimension'] == 'Sector'].copy()
sector_growth_cats_df2 = sector_growth_cats_df2[sector_growth_cats_df2['year'] >= 1999]
sector_growth_cats_df2['stagnant_emp_share'] = sector_growth_cats_df2['stagnant_emp'] / sector_growth_cats_df2['employment']
sector_growth_cats_df2['stagnant_firm_share'] = sector_growth_cats_df2['n_stagnant'] / sector_growth_cats_df2['n_incumbents']

max_year = int(sector_growth_cats_df2['year'].max())

# Order sectors by average employment share (highest first)
sector_order2 = (
    sector_growth_cats_df2.groupby('category')['stagnant_emp_share']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

sector_stagnant_long = sector_growth_cats_df2.melt(
    id_vars=['year', 'category'],
    value_vars=['stagnant_emp_share', 'stagnant_firm_share']
).assign(variable=lambda df: df['variable'].map({
    'stagnant_emp_share': 'Employment share',
    'stagnant_firm_share': 'Firm share'
}))

color_scale = alt.Scale(
    domain=['Employment share', 'Firm share'],
    range=[eco_style.pallete['nominal_1'], eco_style.pallete['nominal_5']]
)

line = alt.Chart().mark_line().encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 4 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('value:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False)),
    color=alt.Color('variable:N', scale=color_scale, legend=alt.Legend(title=None, orient='bottom'))
)

stagnant_sector_chart = alt.layer(line).facet(
    data=sector_stagnant_long,
    facet=alt.Facet('category:N', sort=sector_order2, title=None),
    columns=3
).properties(title='Firm share and employment share in stagnant firms by sector')

stagnant_sector_chart

alt.FacetChart(...)

In [ ]:
# STAGNANT EMPLOYMENT SHARE BY region
# 15 categories — faceted by region, single line per panel

region_growth_cats_df = growth_cats_df[growth_cats_df['dimension'] == 'region'].copy()
region_growth_cats_df = region_growth_cats_df[region_growth_cats_df['year'] >= 1999]
region_growth_cats_df['stagnant_emp_share'] = region_growth_cats_df['stagnant_emp'] / region_growth_cats_df['employment']

max_year = int(region_growth_cats_df['year'].max())

# Order regions by average employment share (highest first)
region_order = (
    region_growth_cats_df.groupby('category')['stagnant_emp_share']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

line = alt.Chart().mark_line(color=eco_style.pallete['nominal_1']).encode(
    x=alt.X('year:O', axis=alt.Axis(labelExpr="datum.value % 4 == 0 ? datum.label : ''", labelAngle=0, title=None)),
    y=alt.Y('stagnant_emp_share:Q', axis=alt.Axis(format='%', title=None), scale=alt.Scale(zero=False))
)

emp_stagnation_region_chart = alt.layer(line).facet(
    data=region_growth_cats_df,
    facet=alt.Facet('category:N', sort=region_order, title=None),
    columns=3
).properties(title='Employment share in stagnant firms by region')

emp_stagnation_region_chart

alt.FacetChart(...)


## **FACT 11. Site expansion: post-crash reduction.**

- Another way firms expand is through opening new physical sites.
- Nearly half of all employees work in a firm that operates multiple sites.
- The overall rate of site expansion has fallen dramatically.
- The likelihood that a firm operates multiple sites differs across industries.
- These findings contrast with evidence of geographic expansion seen in the US.
- Site expansion is inherently local.

In [ ]:
# HEADLINE: Just site expansion
# Site expansion and contraction rates over time
total_site_dynamics_df = site_dynamics_df[site_dynamics_df['dimension'] == 'Total'].copy()

site_dynamics_rates_df = total_site_dynamics_df.melt(
    id_vars=['year'],
    value_vars=['site_exp_rate_entrants', 'site_exp_rate_incumbents'],
    var_name='flow_type', value_name='rate'
)

flow_type_map = {
    'site_exp_rate_entrants': 'Expansion - entrants',
    'site_exp_rate_incumbents': 'Expansion - incumbents'
}
site_dynamics_rates_df['flow_type'] = site_dynamics_rates_df['flow_type'].map(flow_type_map)

color_scale = alt.Scale(
    domain=['Expansion - incumbents', 'Expansion - entrants'],
    range=['#179fdb', '#122b39']
)

# Zero line
zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
    color='black', strokeWidth=0.5
).encode(y='y:Q')

bars = alt.Chart(site_dynamics_rates_df).mark_bar().encode(
    x=alt.X('year:O', axis=alt.Axis(
        labelExpr="datum.value % 2 == 0 ? datum.label : ''",
        labelAngle=0, title=None)),
    y=alt.Y('rate:Q', axis=alt.Axis(format='%'), title="Site expansion rate"),
    color=alt.Color('flow_type:N', scale=color_scale,
                    legend=alt.Legend(title=None, orient='none',
                                       legendX=450,   
                                       legendY=12,  
                                       direction='vertical')),
    order=alt.Order('flow_type:N')
)

site_dynamics_chart = (zero + bars).properties(width=600, height=400)

display(site_dynamics_chart)
#site_dynamics_chart.save(chart_path / 'Descriptive paper/Dynamism/total_site_expansion_rates.png', scale_factor=2.0)
site_dynamics_chart.save(chart_path / 'Exploratory/total_site_expansion_rates.png', scale_factor=2.0)

## **FACT 12. Productivity dispersion has risen.**

In [ ]:

# Clean variable name mapping
label_map = {
    'p10_prod': '10th percentile',
    'p25_prod': '25th percentile',
    'p50_prod': 'Median',
    'p75_prod': '75th percentile',
    'p90_prod': '90th percentile'
}

# Ordered list for legend
percentile_order = [
    '10th percentile',
    '25th percentile',
    'Median',
    '75th percentile',
    '90th percentile'
]

# Publication-quality colour palette (light to dark, colourblind-friendly)
colour_palette = ['#c6dbef', '#6baed6', '#2171b5', '#084594', '#08306b']

# Prep data
size_prod_dispersion = prod_df[prod_df['dimension'] == 'Size continuous']
size_prod_dispersion = size_prod_dispersion[
    size_prod_dispersion['year'].isin([2000, 2010, 2023])
]
size_prod_dispersion = size_prod_dispersion.melt(
    id_vars=['year', 'category'],
    value_vars=['p10_prod', 'p25_prod', 'p50_prod', 'p75_prod', 'p90_prod']
)

# Apply readable labels
size_prod_dispersion['variable'] = size_prod_dispersion['variable'].map(label_map)

chart = alt.Chart(size_prod_dispersion).mark_point(
    filled=True,
    size=40,
    opacity=0.85
).encode(
    x=alt.X(
        'category:O',
        title='Number of employees',
        axis=alt.Axis(
            labelAngle=0,
            labelFontSize=10,
            titleFontSize=11,
            titleFontWeight='normal',
            labelExpr="datum.value == '1' || datum.value == '5' || datum.value == '10' || datum.value == '15' || datum.value == '20' || datum.value == '25+' ? datum.label : ''"
        )
    ),
    y=alt.Y(
        'value:Q',
        title=None,
        scale=alt.Scale(zero=False),
        axis=alt.Axis(
            labelFontSize=10,
            titleFontSize=11,
            titleFontWeight='normal',
            grid=True,
            gridOpacity=0.3
        )
    ),
    color=alt.Color(
        'variable:N',
        title='Productivity percentile',
        sort=percentile_order,
        scale=alt.Scale(
            domain=percentile_order,
            range=colour_palette
        ),
        legend=alt.Legend(
            orient='bottom',
            titleFontSize=11,
            labelFontSize=10,
            symbolSize=80
        )
    ),
    facet=alt.Facet(
        'year:O',
        title=None,
        header=alt.Header(
            labelFontSize=12,
            labelFontWeight='bold',
            labelPadding=8
        )
    )
).properties(
    width=300,
    height=400
).configure_view(
    strokeWidth=0
).configure_axis(
    domainColor='#cccccc',
    tickColor='#cccccc'
).configure_facet(
    spacing=25
)

chart.save(
    chart_path / 'Descriptive paper/Productivity/firmsize_productivity.png',
    scale_factor=3.0  # Higher for publication resolution
)

chart